# 175 — MLP Neuron Profiling

Profile individual MLP neurons: activation statistics, logit impact,
selectivity patterns, correlation clusters, and dead neuron detection.

## Why This Matters

MLP neuron provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_neuron_profiling import (
    neuron_activation_profile,
    neuron_logit_impact,
    neuron_selectivity,
    neuron_correlation_clusters,
    dead_neuron_analysis,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = neuron_activation_profile(model, tokens, layer=0, top_k=5)
print(f"Active: {result['n_active']}/{result['n_total']}  "
      f"Mean act: {result['mean_activation']:.4f}\n")
for n in result['top_neurons']:
    acts = ', '.join(f'{a:.3f}' for a in n['activations'][:5])
    print(f"  Neuron {n['neuron']:3d}: mean={n['mean']:.4f}  max={n['max']:.4f}  [{acts}...]")

In [ ]:
result = neuron_logit_impact(model, tokens, layer=0, top_k=5)
print(f"Target token: {result['target_token']}  Position: {result['position']}\n")
for n in result['top_neurons']:
    sign = '+' if n['logit_contribution'] > 0 else ''
    bar = '#' * int(abs(n['logit_contribution']) * 50)
    print(f"  Neuron {n['neuron']:3d}: {sign}{n['logit_contribution']:.4f}  "
          f"act={n['activation']:.4f}  {bar}")

In [ ]:
result = neuron_selectivity(model, tokens, layer=0, top_k=5)
print(f"Mean selectivity: {result['mean_selectivity']:.3f}\n")
print("Most selective:")
for n in result['most_selective']:
    print(f"  Neuron {n['neuron']:3d}: selectivity={n['selectivity']:.3f}")
print("\nLeast selective:")
for n in result['least_selective']:
    print(f"  Neuron {n['neuron']:3d}: selectivity={n['selectivity']:.3f}")

In [ ]:
result = neuron_correlation_clusters(model, tokens, layer=0, threshold=0.5)
print(f"Active neurons: {result['n_active']}  Correlated pairs: {len(result['correlated_pairs'])}\n")
for p in result['correlated_pairs'][:10]:
    print(f"  Neurons {p['neuron_a']:3d} <-> {p['neuron_b']:3d}: r={p['correlation']:.3f}")

In [ ]:
result = dead_neuron_analysis(model, tokens, layer=0)
print(f"Total neurons: {result['total_neurons']}")
print(f"Dead:      {result['n_dead']} ({result['dead_fraction']:.1%})")
print(f"Near-dead: {result['n_near_dead']}")
print(f"Healthy:   {result['n_healthy']}")
if result['dead_indices']:
    print(f"\nDead neuron indices: {result['dead_indices'][:20]}")